# Modulo de Video - Tech Challenge Fase 4 (Google Colab)

Roda o pipeline de analise de video (postura com MediaPipe/OpenPose + eventos com YOLOv8) sem depender da maquina local.

**Fluxo:** clonar o repo -> instalar -> enviar um video -> processar -> ver e baixar o alerta e o video anotado.

> Dica: em *Ambiente de execucao > Alterar o tipo de ambiente*, **CPU ja basta** (o MediaPipe roda em CPU). GPU nao e necessaria.

## 1. Verificar o ambiente

In [ ]:
import sys, platform
print('Python:', sys.version)
print('SO:', platform.platform())
!nvidia-smi -L 2>/dev/null || echo 'Sem GPU (tudo bem, o pipeline roda em CPU).'

## 2. Clonar o repositorio

Se ja estiver clonado, apenas atualiza (git pull).

In [ ]:
import os
REPO = '/content/modulo_video'
if not os.path.exists(REPO):
    !git clone https://github.com/marcoswoc/modulo_video.git {REPO}
else:
    !cd {REPO} && git pull
%cd {REPO}

## 3. Instalar as dependencias

Inclui o `ultralytics` (YOLOv8) e o `mediapipe`. Pode levar 1 a 2 minutos.

In [ ]:
!pip install -q -r requirements.txt
# No Colab (Python 3.12) o mediapipe tenta importar o tensorflow pre-instalado e
# quebra por conflito de protobuf. Como este modulo NAO usa tensorflow, removemos
# para o mediapipe carregar normalmente (a pose estimation nao precisa dele).
!pip uninstall -y tensorflow tensorflow-cpu 2>/dev/null
print('Dependencias instaladas.')

## 4. Enviar um video

Escolha um arquivo de video (.mp4) da sua maquina. Ele sera salvo em `data/entrada/`.

> Dica: para a demo ficar rapida, use um trecho curto (poucos segundos) e resolucao ate 720p.

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('data/entrada', exist_ok=True)
enviados = files.upload()
nome_video = list(enviados.keys())[0]
destino = os.path.join('data/entrada', nome_video)
shutil.move(nome_video, destino)
print('Video salvo em:', destino)

## 5. Rodar o pipeline

Gera o **alerta JSON** e um **video anotado** com o esqueleto desenhado.

> Se o video for pesado e demorar demais, rode a mesma linha adicionando `--sem-objetos` para desativar o YOLOv8.

In [ ]:
PATIENT_ID = 'video_001'
import os
os.makedirs('data/saida', exist_ok=True)
saida_json = 'data/saida/' + PATIENT_ID + '.json'
saida_anotado = 'data/saida/' + PATIENT_ID + '_anotado.mp4'
!python main.py --video "{destino}" --patient-id {PATIENT_ID} --saida "{saida_json}" --anotado "{saida_anotado}"

## 6. Ver o alerta gerado (schema oficial do grupo)

In [ ]:
import json
with open(saida_json, encoding='utf-8') as f:
    alerta = json.load(f)
print(json.dumps(alerta, ensure_ascii=False, indent=2))

## 7. Visualizar o video anotado (inline)

Funciona melhor com videos curtos. Para videos grandes, use a celula de download abaixo.

In [ ]:
import os, subprocess
from IPython.display import HTML, display
from base64 import b64encode

if not (os.path.exists(saida_anotado) and os.path.getsize(saida_anotado) > 0):
    print('Video anotado nao foi gerado (verifique a etapa 5).')
else:
    # O OpenCV grava em mp4v, que o navegador nao toca inline. Reencoda para
    # H264 (libx264) com o ffmpeg do Colab para exibir dentro do notebook.
    saida_web = saida_anotado.replace('.mp4', '_web.mp4')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', saida_anotado,
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', saida_web], check=False)
    alvo = saida_web if (os.path.exists(saida_web) and os.path.getsize(saida_web) > 0) else saida_anotado
    dados = open(alvo, 'rb').read()
    if len(dados) > 40_000_000:
        print('Video grande demais para exibir inline. Use a celula de download (8).')
    else:
        url = 'data:video/mp4;base64,' + b64encode(dados).decode()
        display(HTML('<video width=480 controls><source src="' + url + '" type="video/mp4"></video>'))

## 8. Baixar os resultados

Baixa o JSON do alerta e o video anotado para a sua maquina (uteis no relatorio e na apresentacao).

In [ ]:
from google.colab import files
import os
files.download(saida_json)
if os.path.exists(saida_anotado) and os.path.getsize(saida_anotado) > 0:
    files.download(saida_anotado)

## Observacoes

- **CPU basta**: o gargalo (MediaPipe + YOLOv8) roda em CPU. GPU acelera pouco aqui.
- **Codec**: no Colab (Linux) o video anotado sai em H264 (avc1), que toca inline no notebook. O codigo escolhe o codec automaticamente por sistema.
- **Conflito mediapipe x tensorflow (Colab)**: ao rodar, o mediapipe importa o tensorflow pre-instalado do Colab e pode quebrar por conflito de protobuf (`cannot import name 'runtime_version'`). A celula de instalacao (3) ja remove o tensorflow para evitar isso. Se ainda ocorrer, rode `!pip uninstall -y tensorflow` e execute a celula do pipeline de novo.
- **Desempenho**: para acelerar, use videos curtos/720p, ou desative o YOLOv8 com `--sem-objetos`.